# Ternary Bonsai 27B Server

**27B dense at 1.71-bit ternary** — Qwen3.6-27B with {−1, 0, +1} weights by PrismML.

- **~3.5 GB** model file — 9.4× smaller than FP16, 4× smaller than Q4_K_M
- Only ~4 GB VRAM — massive headroom for long context on T4 (32K+)
- Qwen3.6 base (1M context, thinking mode, Apache 2.0)
- **Needs custom PrismML llama.cpp fork** (ternary CUDA kernels not upstream yet)

**⚠️ Build time**: This notebook compiles llama.cpp from source on the T4.
Expect **30-45 minutes** for the build (T4 has only 2 CPU cores).
Once built, the binary is cached in `/tmp` — re-running skips the build.

**Pre-built binary** (faster): A pre-built CUDA 12.4 binary for T4 will be available
at the GitHub Releases page. Until then, the notebook builds from source.

In [ ]:
# 1. Check GPU & install deps
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install -q huggingface_hub
import os, subprocess, sys, time, re, threading
print("Ready.")

In [ ]:
# 2. Clone & build PrismML llama.cpp fork
# Only builds llama-server (OpenAI-compatible HTTP API).
# Build is cached — if /tmp/prism-llama.cpp/build/bin/llama-server exists, skips.

PRISM_DIR = "/tmp/prism-llama.cpp"
BUILD_DIR = f"{PRISM_DIR}/build"
LLAMA_SERVER = f"{BUILD_DIR}/bin/llama-server"

# Try pre-built binary first
PREBUILT_URL = "https://github.com/phantomic12/colab-llm-server/releases/latest/download/llama-server-prism-cuda-T4.tar.zst"

if not os.path.exists(LLAMA_SERVER):
    # Attempt pre-built download
    try:
        import urllib.request
        print("Trying pre-built binary...")
        urllib.request.urlretrieve(PREBUILT_URL, "/tmp/llama-server-prism.tar.zst")
        !mkdir -p /tmp/prism-prebuilt && cd /tmp/prism-prebuilt && tar --zstd -xf /tmp/llama-server-prism.tar.zst
        # Find the binary
        for root, dirs, files in os.walk("/tmp/prism-prebuilt"):
            for f in files:
                if f == "llama-server":
                    os.makedirs(os.path.dirname(LLAMA_SERVER), exist_ok=True)
                    os.rename(os.path.join(root, f), LLAMA_SERVER)
                    # Copy libs
                    for lib in files:
                        if lib.endswith(".so") or lib.endswith(".so.1"):
                            os.rename(os.path.join(root, lib), f"{BUILD_DIR}/bin/{lib}")
                    print("Pre-built binary installed!")
                    break
    except Exception as e:
        print(f"Pre-built not available ({e}), building from source...")
        
if not os.path.exists(LLAMA_SERVER):
    # Build from source
    if not os.path.exists(PRISM_DIR):
        print("Cloning PrismML llama.cpp fork...")
        !git clone --depth 1 https://github.com/PrismML-Eng/llama.cpp.git {PRISM_DIR}
    
    os.makedirs(BUILD_DIR, exist_ok=True)
    
    if not os.path.exists(f"{BUILD_DIR}/CMakeCache.txt"):
        print("Configuring cmake...")
        !cd {BUILD_DIR} && cmake .. -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_BUILD_TYPE=Release
    
    print("Building llama-server (30-45 min on T4)...")
    t0 = time.time()
    !cd {BUILD_DIR} && cmake --build . --config Release -j2 --target llama-server
    print(f"Build done in {time.time()-t0:.0f}s")

!ls -lh {LLAMA_SERVER}
assert os.path.exists(LLAMA_SERVER), "Build failed!"
print("llama-server ready.")

In [ ]:
# 3. Download Ternary Bonsai 27B model (~3.5 GB)
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="prism-ml/Ternary-Bonsai-27B-gguf",
    filename="Ternary-Bonsai-27B-Q2_0.gguf"
)
size_gb = os.path.getsize(model_path) / 1e9
print(f"Model: {os.path.basename(model_path)} ({size_gb:.2f} GB)")

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

MODEL_NAME = "ternary-bonsai-27b"
CONTEXT = 32768
PORT = 8080

In [ ]:
# 4. Start Cloudflare tunnel
TUNNEL_URL = None

def run_tunnel(port=PORT):
    global TUNNEL_URL
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="")
        m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
        if m and not TUNNEL_URL:
            TUNNEL_URL = m.group(1)
            print(f"\n>>> TUNNEL URL: {TUNNEL_URL}")
            print(f">>> API: {TUNNEL_URL}/v1/chat/completions")

thread = threading.Thread(target=run_tunnel, daemon=True)
thread.start()

for _ in range(30):
    if TUNNEL_URL:
        break
    time.sleep(1)

if not TUNNEL_URL:
    print("WARNING: Tunnel URL not detected.")
else:
    print(f"\nReady! API at: {TUNNEL_URL}/v1/chat/completions")

In [ ]:
# 5. Start llama-server (PrismML fork)
# llama-server provides an OpenAI-compatible HTTP API directly.

!pkill -f llama-server || true

LD_LIB = BUILD_DIR
cmd = [
    LLAMA_SERVER,
    "-m", model_path,
    "-ngl", "99",
    "-c", str(CONTEXT),
    "-t", "2",
    "--host", "0.0.0.0",
    "--port", str(PORT),
]

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = LD_LIB

server_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=env
)

print(f"Started llama-server (PID {server_proc.pid})")

# Wait for server to be ready (health endpoint)
for i in range(120):
    time.sleep(2)
    try:
        r = subprocess.run(
            ["curl", "-s", f"http://localhost:{PORT}/health"],
            capture_output=True, text=True, timeout=5
        )
        if r.returncode == 0 and ("ok" in r.stdout.lower() or '"status"' in r.stdout):
            print(f"Server ready after {i*2}s!")
            break
    except:
        pass
    # Show progress every 30s
    if i > 0 and i % 15 == 0:
        line = server_proc.stdout.readline()
        if line:
            print(f"  [{i*2}s] {line.rstrip()[:120]}")
else:
    print("Server may still be loading — check output above.")
    for _ in range(3):
        line = server_proc.stdout.readline()
        if line:
            print(f"  {line.rstrip()[:200]}")

In [ ]:
# 6. Test the API
print("curl test:")
!curl -s {TUNNEL_URL}/v1/chat/completions \
  -X POST -H "Content-Type: application/json" \
  -d '{"messages":[{"role":"user","content":"What is 17 * 23? Think step by step."}],"max_tokens":200}'

print("\n\n--- Python client test ---")
from openai import OpenAI
client = OpenAI(base_url=f"{TUNNEL_URL}/v1", api_key="not-needed")
r = client.chat.completions.create(
    model="ternary-bonsai-27b",
    messages=[{"role": "user", "content": "Say hello in 3 languages."}],
    max_tokens=100
)
print(r.choices[0].message.content)

print("\nVRAM usage:")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader
print("\nDone! API available at:", TUNNEL_URL, "/v1")